In [1]:
!pip install --upgrade google-generativeai pandas

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import pandas as pd
import numpy as np
import vertexai
import warnings
from vertexai.generative_models import GenerativeModel
from vertexai.language_models import TextEmbeddingModel
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
import time

In [4]:
PROJECT_ID = "datahub-3c396"
REGION = "us-central1"

In [5]:
vertexai.init(project=PROJECT_ID, location=REGION)
model = GenerativeModel("gemini-2.5-pro")
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

response = model.generate_content("Say hello!")
print(response.text)

Hello there! How can I help you today?


In [5]:
# Merging datasets together
df1 = pd.read_csv("facility_maintenance_logs.csv")
df2 = pd.read_csv("management_dataset.csv")

merged_logs = pd.concat([df1, df2], ignore_index=True)
merged_logs.to_csv("merged_logs.csv", index=False)

In [6]:
df_logs = pd.read_csv("merged_logs.csv")

df_logs["log_text"] = df_logs["log_text"].astype(str)

print("Operational memory loaded:", len(df_logs))
df_logs.head()

Operational memory loaded: 1002


,log_text,location,category,severity,action_taken,date,"log_text,location,category,severity,action_taken,date"
0,Network issue at Sports Hall Court 1: WiFi out...,Sports Hall Court 1,Network,High,Restarted access point,2024-04-12,NaN
1,Mechanical issue at Main Library: unusual vibr...,Main Library,Mechanical,Medium,Reset control system,2024-04-09,NaN
2,Electrical issue at Lecture Hall A: circuit br...,Lecture Hall A,Electrical,Medium,Replaced fuse,2024-06-06,NaN
3,Plumbing issue at Main Library: slow drainage.,Main Library,Plumbing,High,Sealed leaking joint,2024-03-12,NaN
4,Mechanical issue at Lecture Hall A: unusual vi...,Lecture Hall A,Mechanical,High,Replaced worn relay,2024-04-18,NaN


In [7]:
def get_embeddings(texts):
    embeddings = embedding_model.get_embeddings(texts)
    return [e.values for e in embeddings]

all_embeddings = []
batch_size = 20
texts = df_logs["log_text"].tolist()

In [8]:
print("Generating embeddings...")

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    
    try:
        batch_results = get_embeddings(batch)
        all_embeddings.extend(batch_results)
        print(f"Processed {i + len(batch)} / {len(texts)}")
        time.sleep(4)
    except Exception as e:
        print(f"Quota Hit or Error at index {i}: {e}")
        print("Waiting 60 seconds to reset...")
        time.sleep(60)
        batch_results = get_embeddings(batch)
        all_embeddings.extend(batch_results)

ALL_EMBEDDINGS = np.array(all_embeddings)

print("Operational memory indexed: ", len(ALL_EMBEDDINGS))

Generating embeddings...
Processed 20 / 1002
Processed 40 / 1002
Processed 60 / 1002
Processed 80 / 1002
Processed 100 / 1002
Processed 120 / 1002
Processed 140 / 1002
Processed 160 / 1002
Processed 180 / 1002
Processed 200 / 1002
Processed 220 / 1002
Processed 240 / 1002
Processed 260 / 1002
Processed 280 / 1002
Processed 300 / 1002
Processed 320 / 1002
Processed 340 / 1002
Processed 360 / 1002
Processed 380 / 1002
Processed 400 / 1002
Processed 420 / 1002
Processed 440 / 1002
Processed 460 / 1002
Processed 480 / 1002
Processed 500 / 1002
Processed 520 / 1002
Processed 540 / 1002
Processed 560 / 1002
Processed 580 / 1002
Processed 600 / 1002
Processed 620 / 1002
Processed 640 / 1002
Processed 660 / 1002
Processed 680 / 1002
Processed 700 / 1002
Processed 720 / 1002
Processed 740 / 1002
Processed 760 / 1002
Processed 780 / 1002
Processed 800 / 1002
Processed 820 / 1002
Processed 840 / 1002
Processed 860 / 1002
Processed 880 / 1002
Processed 900 / 1002
Processed 920 / 1002
Processed 940

In [10]:
def find_similar(new_issue, top_k=5):
    query_embeddings = get_embeddings([new_issue])[0]

    similarities = cosine_similarity(
        [query_embeddings],
        ALL_EMBEDDINGS
    )[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    result = []
    
    for idx in top_indices:
        result.append({
            "log_text": df_logs.iloc[idx]["log_text"],
            "location": df_logs.iloc[idx]["location"],
            "category": df_logs.iloc[idx]["category"],
            "action_taken": df_logs.iloc[idx]["action_taken"],
            "similarity_score": float(similarities[idx])
        })

    return result

In [11]:
def calculate_recurrence(similar_incidents, similarity_threshold=0.58):
    strong_matches = [
        s for s in similar_incidents
        if s["similarity_score"] > similarity_threshold
    ]

    return len(strong_matches)

In [12]:
def generate_direct_solution(user_name, new_issue, similar_incidents, recurrence_count, inventory, bookings, notes, chat_history):
    if recurrence_count == 0:
        return {
            "user_message": f"""
    Issue Logged: {new_issue}

    Status: No strong historical matches found.
    Action: Escalated for manual review.
    
    Please provide:
    - Exact location
    - Equipment involved
    - Severity level
    """
        }
    
    prompt = f"""
    You are an Operational Intelligence Engine, for a pickleball club.

    A user named {user_name} reported:
    {new_issue}
    
    Top Related Historical Records:
    {similar_incidents}

    Current Inventory:
    {inventory}
    
    Current Bookings:
    {bookings}
    
    Internal Staff Notes:
    {notes}
    
    Recent Chat History:
    {chat_history}

    Based on the reccuring pattern, synthesize the MOST effective solution. 
    
    Rules: 
    - provide a clear, professional message. 
    - if inventory quantity is 0, recommend reorder.
    - if note mentions supplier timing, consider it.
    - greet the user. 
    - provide 1-3, or more clear action steps. 
    - do not mention similarity scores. - do not output JSON. 
    - keep it concise but confident. 
    
    Return ONLY final message. 
    """ 
    
    response = model.generate_content(prompt) 
    return response.text
    

In [13]:
def process_new_incident(user_name, new_issue):
    text = new_issue.lower().strip()
    greetings = ["hi", "hello", "hey"]

    if text in greetings:
        return {
            "user_message":f"Hello {user_name}! How can I assist you today?"
        }

    similar_incidents = find_similar(new_issue)
    recurrence_count = calculate_recurrence(similar_incidents) 
    
    inventory = get_inventory()
    bookings = get_bookings()
    notes = get_notes()
    chat_history = get_recent_chat()
    
    if recurrence_count == 0:
        prompt = f"""
        You are a helpful operational assistant for a pickball club..

        Current Inventory: {inventory}
        Current Bookings: {bookings}
        Internal Notes: {notes}
        Recent Chat: {chat_history}

        User {user_name} said "{new_issue}"

        If relevant, use the operational data above.
        Otherwise, respond professionally and ask clarifying questions if needed.
        """
        response = model.generate_content(prompt)

        return {
            "user_message": response.text
        }

    user_message = generate_direct_solution(
        user_name,
        new_issue,
        similar_incidents,
        recurrence_count,
        inventory,
        bookings,
        notes,
        chat_history
    )
    
    return {
        "new_incident": new_issue,
        "recurrence_count": recurrence_count,
        "similar_incidents": similar_incidents,
        "user_message": user_message
    }

### Simple Testing

In [13]:
# test_issue = "Air conditioning in Lecture Hall A not cooling again."

# result = process_new_incident("John",test_issue)
# print(result["user_message"])

In [3]:
!pip install flask

In [15]:
!pip install flask-cors

In [19]:
!pip install firebase-admin

In [14]:
import firebase_admin
from firebase_admin import credentials, firestore

cred = credentials.Certificate("serviceAccountKey.json")
firebase_admin.initialize_app(cred)

db = firestore.client()

In [15]:
def get_inventory():
    docs = db.collection("inventory").stream()
    return [doc.to_dict() for doc in docs]

def get_bookings():
    docs = db.collection("bookings").stream()
    return [doc.to_dict() for doc in docs]

def get_notes():
    docs = db.collection("notes").stream()
    return [doc.to_dict() for doc in docs]

def get_recent_chat(limit=5):
    docs = db.collection("chat_history") \
             .order_by("timestamp") \
             .limit_to_last(limit) \
             .get()
    return [doc.to_dict() for doc in docs]

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route("/process_incident", methods=["POST"])
def process_incident():
    print("Request received")
    data = request.json
    print(data)
    
    user_name = data.get("user_name")
    issue_text = data.get("issue_text")

    result = process_new_incident(user_name, issue_text)

    return jsonify({"user_message": result["user_message"]})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5050, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5050
 * Running on http://10.128.0.2:5050
Press CTRL+C to quit
